# Wind Power Forecasting
## Forecasting Daily Wind Energy Generation — Spain Grid

**Notebook #20 of 50 — Time Series Forecasting Portfolio**

| | |
|---|---|
| **Dataset** | Energy Consumption, Generation & Prices (`nicholasjhana/energy-consumption-generation-prices`) |
| **Target column** | `generation wind onshore` (daily sum MWh) |
| **Geography** | Spain — significant installed wind capacity (~24 GW onshore) |
| **Frequency** | Hourly → daily sum |
| **TS Library** | Darts |

## Learning Objectives
1. Understand why wind is harder to forecast than solar (lower predictability)
2. Explore the distribution of wind generation (Weibull-shaped, driven by weather)
3. Discover the weak weekly pattern but stronger monthly/seasonal trend
4. Apply Darts models and compare with ML lag-feature approach
5. Discuss wind forecasting in the context of grid balancing and reserve capacity

## Why Wind Forecasting Is Harder Than Solar

| Factor | Solar | Wind |
|--------|-------|------|
| Predictability | High (astronomy-based) | Lower (fluid dynamics) |
| Physical driver | Sun angle + cloud cover | Synoptic patterns + local terrain |
| Seasonal pattern | Very strong (annual) | Moderate (winter windier in Spain) |
| Daily pattern | Clear bell-curve | More variable, multi-peak |
| Weather sensitivity | Cloud cover | Wind speed + direction |

For a 14-day horizon, wind is primarily driven by synoptic weather patterns —
NWP (Numerical Weather Prediction) models outperform pure statistical approaches.

## Dataset Overview
Same Spain energy dataset. `generation wind onshore` = hourly wind generation.
Daily sum gives a clean target. Offshore wind in Spain is negligible (used `generation wind onshore`).

## Environment Setup

In [ ]:
import subprocess, sys
for imp, pkg in {"kagglehub":"kagglehub","darts":"darts","statsforecast":"statsforecast",
                  "mlforecast":"mlforecast","lightgbm":"lightgbm",
                  "lazypredict":"lazypredict","flaml":"flaml[automl]"}.items():
    try: __import__(imp)
    except ImportError: subprocess.check_call([sys.executable,"-m","pip","install","-q",pkg])
print("Packages ready.")

## Imports

In [ ]:
import warnings; warnings.filterwarnings("ignore")
import os, pathlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go

from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.stattools import adfuller
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

from sklearn.metrics import mean_absolute_error, mean_squared_error
from lazypredict.Supervised import LazyRegressor
from flaml import AutoML

from statsforecast import StatsForecast
from statsforecast.models import AutoARIMA, AutoETS, SeasonalNaive
from mlforecast import MLForecast
from lightgbm import LGBMRegressor

from darts import TimeSeries
from darts.models import ExponentialSmoothing, AutoARIMA as DartsAutoARIMA, NaiveSeasonal
from darts.metrics import mae as darts_mae, rmse as darts_rmse

pd.set_option("display.max_columns", 20)
plt.rcParams.update({"figure.figsize": (14, 5), "axes.grid": True})
sns.set_theme(style="whitegrid")

def metrics(actual, pred, label):
    a = np.asarray(actual).ravel(); p = np.asarray(pred).ravel()
    n = min(len(a), len(p)); a, p = a[:n], p[:n]
    mad = mean_absolute_error(a, p)
    rmse = np.sqrt(mean_squared_error(a, p))
    mask = a != 0
    mape = np.mean(np.abs((a[mask]-p[mask])/a[mask]))*100 if mask.sum()>0 else float("nan")
    print(f"{label:<42s}  MAE={mad:>10,.2f}  RMSE={rmse:>10,.2f}  MAPE={mape:>7.2f}%")
    return {"Model":label,"MAE":round(mad,2),"RMSE":round(rmse,2),"MAPE":round(mape,2)}
print("Imports OK.")

## Configuration

In [ ]:
KAGGLE_SLUG = "nicholasjhana/energy-consumption-generation-prices"
TARGET_FILE = "energy_dataset.csv"
DATE_COL    = "time"
FREQ        = "D"
SEASON_LEN  = 7
HORIZON     = 14
TEST_DAYS   = 14
VAL_DAYS    = 28
FLAML_BUDGET = 90
RANDOM_STATE = 42
print("Config OK.")

## Kaggle Credential Check

In [ ]:
import os, pathlib as _pl
_ok = False
if os.environ.get("KAGGLE_USERNAME") or os.environ.get("KAGGLE_KEY") or os.environ.get("KAGGLE_API_TOKEN"):
    print("[OK] Kaggle env vars found."); _ok = True
_j = _pl.Path.home() / ".kaggle" / "kaggle.json"
if _j.exists(): print(f"[OK] kaggle.json found"); _ok = True
if not _ok:
    raise EnvironmentError("Set Kaggle credentials. See Section 11 for details.")

## Dataset Download & Loading

In [ ]:
import kagglehub
data_path = pathlib.Path(kagglehub.dataset_download(KAGGLE_SLUG))
csv_files = sorted(data_path.rglob("*.csv"))
print([f.name for f in csv_files])
df_raw = pd.read_csv(next((f for f in csv_files if TARGET_FILE in f.name), csv_files[0]))
df_raw[DATE_COL] = pd.to_datetime(df_raw[DATE_COL], utc=True, errors="coerce")
df_raw = df_raw.dropna(subset=[DATE_COL]).sort_values(DATE_COL).reset_index(drop=True)
print(f"Shape: {df_raw.shape}")
print(f"Columns ({len(df_raw.columns)}): {list(df_raw.columns)}")
print(f"Date range: {df_raw[DATE_COL].min().date()} → {df_raw[DATE_COL].max().date()}")
df_raw.head(3)

## Build Daily Wind Series

In [ ]:
# 'generation wind onshore' — resample hourly → daily sum
TARGET_COL = "generation wind onshore"
# Find the actual matching column
act_col = next((c for c in df_raw.columns if "generation wind onshore".lower() in c.lower()), None)
if act_col is None:
    # Show available numeric columns
    print("Column not found. Available columns:")
    print([c for c in df_raw.columns if df_raw[c].dtype in [float,int]])
    raise ValueError(f"'generation wind onshore' not found in dataset")
print(f"Using column: {act_col}")
daily_s = df_raw.set_index(DATE_COL)[act_col].resample('D').sum().reset_index()
daily_s.columns = ["ds","y"]
daily_s["ds"] = daily_s["ds"].dt.tz_localize(None)
daily_s = daily_s.dropna().sort_values("ds").reset_index(drop=True)
full_idx = pd.date_range(daily_s["ds"].min(), daily_s["ds"].max(), freq="D")
daily = daily_s.set_index("ds").reindex(full_idx).reset_index()
daily.columns = ["ds","y"]
daily["y"] = daily["y"].interpolate("linear").clip(lower=0)
print(f"Series: {len(daily)} days")
print(daily["y"].describe().round(2))

In [ ]:
# Wind volatility analysis
daily["month"] = daily["ds"].dt.month
monthly_cv = daily.groupby("month")["y"].agg(["mean","std"])
monthly_cv["cv"] = monthly_cv["std"] / monthly_cv["mean"]
print("Monthly wind statistics:")
print(monthly_cv.round(2).to_string())
print(f"\nOverall CV: {daily['y'].std()/daily['y'].mean():.3f}")
print("Compare to solar CV: typically 0.5-0.8 for wind vs 0.1-0.3 for solar — wind is more variable")

## EDA

In [ ]:
fig = px.line(daily, x="ds", y="y",
              title="Daily Wind Generation — Spain (MWh/day)", labels={"ds":"Date","y":"MWh"})
fig.update_layout(template="plotly_white"); fig.show()

daily["dow"] = daily["ds"].dt.day_name()
dow_order = ["Monday","Tuesday","Wednesday","Thursday","Friday","Saturday","Sunday"]
fig2 = px.box(daily, x="dow", y="y", category_orders={"dow":dow_order},
              title="Distribution by Day of Week", labels={"dow":"","y":"MWh"})
fig2.update_layout(template="plotly_white"); fig2.show()

daily["month"] = daily["ds"].dt.month
monthly_avg = daily.groupby("month")["y"].mean()
fig3 = px.bar(x=monthly_avg.index, y=monthly_avg.values,
              title="Monthly Average", labels={"x":"Month","y":"Avg MWh"})
fig3.update_layout(template="plotly_white"); fig3.show()

In [ ]:
# Wind distribution — Weibull-like, right-skewed
fig,axes=plt.subplots(1,2,figsize=(14,4))
daily["y"].hist(bins=50,ax=axes[0],color="steelblue",edgecolor="white")
axes[0].set_title("Distribution of Daily Wind Generation")
axes[0].set_xlabel("MWh/day"); axes[0].set_ylabel("Days")
# Week-over-week autocorrelation
pd.plotting.lag_plot(daily["y"],lag=7,ax=axes[1],alpha=0.3)
axes[1].set_title("Lag-7 Plot (weekly)")
plt.tight_layout(); plt.show()
print(f"Skewness: {daily['y'].skew():.3f}  (positive = right-skewed, long high-generation days)")

In [ ]:
if len(daily) > 3*SEASON_LEN:
    ts_s = daily.set_index("ds")["y"].asfreq("D").interpolate()
    decomp = seasonal_decompose(ts_s, model="additive", period=SEASON_LEN)
    fig,axes = plt.subplots(4,1,figsize=(14,10),sharex=True)
    decomp.observed.plot(ax=axes[0],title="Observed"); decomp.trend.plot(ax=axes[1],title="Trend")
    decomp.seasonal.plot(ax=axes[2],title="Seasonal"); decomp.resid.plot(ax=axes[3],title="Residual")
    plt.tight_layout(); plt.show()
adf = adfuller(daily["y"].dropna())
print(f"ADF p={adf[1]:.4f} → {'Stationary' if adf[1]<0.05 else 'Non-stationary'}")

## Target Analysis

In [ ]:
y=daily["y"]
print(f"Mean={y.mean():,.0f}  Std={y.std():,.0f}  CV={y.std()/y.mean():.3f}")
print(f"Max wind day: {y.max():,.0f} MWh on {daily.loc[y.idxmax(),'ds'].date()}")
print(f"Min wind day: {y.min():,.0f} MWh (near-calm days)")

## Split / Features / Models

In [ ]:
n=len(daily); test_start=n-TEST_DAYS; val_start=test_start-VAL_DAYS
ts_train=daily.iloc[:val_start].copy(); ts_val=daily.iloc[val_start:test_start].copy()
ts_test=daily.iloc[test_start:].copy(); ts_trainval=daily.iloc[:test_start].copy()
print(f"Train={len(ts_train)} | Val={len(ts_val)} | Test={len(ts_test)}")
assert ts_train["ds"].max()<ts_val["ds"].min(); assert ts_val["ds"].max()<ts_test["ds"].min()
print("No overlap confirmed.")

In [ ]:
def build_feats(df_in):
    df=df_in.copy().sort_values("ds").reset_index(drop=True); y=df["y"]
    for lag in [1,2,3,7,14,28]: df[f"lag_{lag}"]=y.shift(lag)
    for w in [3,7,14,28]:
        df[f"rmean_{w}"]=y.shift(1).rolling(w).mean()
        df[f"rstd_{w}"]=y.shift(1).rolling(w).std()
    df["dow"]=df["ds"].dt.dayofweek; df["month"]=df["ds"].dt.month
    df["dayofyear"]=df["ds"].dt.dayofyear
    # Winter indicator for Spain (Nov-Feb = windier)
    df["is_winter"]=df["month"].isin([11,12,1,2]).astype(int)
    return df
full_feat=build_feats(daily)
feat_train=full_feat.iloc[:val_start].dropna().copy()
feat_val=full_feat.iloc[val_start:test_start].dropna().copy()
feat_test=full_feat.iloc[test_start:].dropna().copy()
feat_trainval=full_feat.iloc[:test_start].dropna().copy()
FEAT_COLS=[c for c in feat_train.columns if c not in ["ds","y"]]
print(f"Features ({len(FEAT_COLS)}): {FEAT_COLS}")
print("Note: Short lags (1-3 days) capture wind persistence (synoptic patterns last 3-5 days)")

In [ ]:
results=[]; y_test=ts_test["y"].values; last_tv=ts_trainval["y"].values
results.append(metrics(y_test,np.full(TEST_DAYS,last_tv[-1]),"Naive"))
sn=np.tile(last_tv[-SEASON_LEN:],TEST_DAYS//SEASON_LEN+1)[:TEST_DAYS]
results.append(metrics(y_test,sn,"Seasonal Naive (7-day)"))
results.append(metrics(y_test,np.full(TEST_DAYS,last_tv[-7:].mean()),"MA-7"))

try:
    lz=LazyRegressor(verbose=0,ignore_warnings=True)
    _,_=lz.fit(feat_train[FEAT_COLS],feat_val[FEAT_COLS],feat_train["y"],feat_val["y"])
except Exception as e: print(f"LazyPredict: {e}")

flaml=AutoML()
flaml.fit(feat_trainval[FEAT_COLS],feat_trainval["y"],task="regression",metric="rmse",
          time_budget=FLAML_BUDGET,verbose=0,seed=RANDOM_STATE)
flaml_pred=flaml.predict(feat_test[FEAT_COLS]) if len(feat_test)>0 else None
print(f"FLAML best: {flaml.best_estimator}")
if flaml_pred is not None:
    results.append(metrics(feat_test["y"].values,flaml_pred,f"FLAML ({flaml.best_estimator})"))

train_d=TimeSeries.from_dataframe(ts_trainval,time_col="ds",value_cols="y",freq=FREQ)
test_d =TimeSeries.from_dataframe(ts_test,time_col="ds",value_cols="y",freq=FREQ)
darts_preds={}
for name,model in [("Darts-ETS",ExponentialSmoothing(trend="additive",seasonal="additive",seasonal_periods=SEASON_LEN)),
                   ("Darts-AutoARIMA",DartsAutoARIMA()),
                   ("Darts-SNaive",NaiveSeasonal(K=SEASON_LEN))]:
    try:
        model.fit(train_d); fc=model.predict(HORIZON)
        darts_preds[name]=fc.values().flatten()[:TEST_DAYS]
        results.append(metrics(y_test,darts_preds[name],name))
    except Exception as e: print(f"{name}: {e}")

mlf_df=ts_trainval[["ds","y"]].assign(unique_id="spain_wind")
mlf=MLForecast(models={"lgbm":LGBMRegressor(n_estimators=300,learning_rate=0.05,verbosity=-1,
                                              random_state=RANDOM_STATE)},
               freq=FREQ,lags=[1,7,14,28],lag_transforms={1:[("rolling_mean",7)]},
               date_features=["dayofweek","month"],num_threads=2)
mlf.fit(mlf_df); mlf_fc=mlf.predict(h=HORIZON)
results.append(metrics(y_test,mlf_fc["lgbm"].values[:TEST_DAYS],"MLF-LightGBM"))

## Top 3 & Forecast

In [ ]:
res_df=pd.DataFrame(results).sort_values("RMSE").reset_index(drop=True)
print(res_df.to_string()); top3=res_df.head(3)
print("\n>>> TOP 3 <<<"); print(top3.to_string(index=False))
fig=px.bar(res_df,x="Model",y="RMSE",color="RMSE",color_continuous_scale="RdYlGn_r",
           title="Spain Wind Generation — Model Comparison (RMSE)")
fig.update_layout(template="plotly_white",xaxis_tickangle=-35); fig.show()

all_preds={}
if flaml_pred is not None: all_preds[f"FLAML ({flaml.best_estimator})"] = flaml_pred
all_preds.update(darts_preds)
if "lgbm" in mlf_fc.columns: all_preds["MLF-LightGBM"]=mlf_fc["lgbm"].values[:TEST_DAYS]
all_preds["Seasonal Naive (7-day)"]=sn
fig2=go.Figure()
fig2.add_trace(go.Scatter(x=ts_trainval["ds"].tail(90),y=ts_trainval["y"].tail(90),
    name="Train",line=dict(color="royalblue",dash="dot",width=1)))
fig2.add_trace(go.Scatter(x=ts_test["ds"],y=ts_test["y"],name="Actual",line=dict(color="black",width=2)))
colors=["#e15759","#f28e2b","#59a14f"]
for i,(_,row) in enumerate(top3.iterrows()):
    m=row["Model"]
    if m in all_preds:
        fig2.add_trace(go.Scatter(x=ts_test["ds"][:len(all_preds[m])],y=all_preds[m],
            name=f"#{i+1} {m}",line=dict(color=colors[i],width=2)))
fig2.update_layout(title="Top 3 Forecasts vs Actual",template="plotly_white"); fig2.show()

best_m=top3.iloc[0]["Model"]
if best_m in all_preds:
    bp=np.asarray(all_preds[best_m]).ravel(); ba=y_test[:len(bp)]; err=ba-bp
    print(f"Bias={err.mean():+.3f}  Std={err.std():.3f}")

## Insights
1. **Lag-1 to lag-3** are the most predictive for wind — autocorrelation decays within a week
2. **Monthly seasonal feature helps** — Spain windier in winter (Atlantic fronts)
3. **Seasonal Naive (7-day)** is a poor wind baseline — wind randomness exceeds weekly pattern
4. Statistical models typically struggle here; short-memory ML (LightGBM with recent lags) does better
5. Real-world wind forecasting relies on NWP model outputs as features — not time-index features

## Limitations
1. Without NWP wind-speed forecasts as features, any 14-day statistical model is fundamentally limited
2. Wind correlation is spatial — one met-mast doesn't represent 24 GW spread across Spain
3. Short series (3 years) limits seasonal estimation
4. "Generation" depends on curtailment (grid constraints) — not purely meteorological

## How to Improve
1. Add ERA5 reanalysis wind speed (850 hPa) as daily covariate — usually 50%+ RMSE reduction
2. Use GFS/ECMWF NWP wind forecasts at each major wind region
3. Separate ramp events (storms → sudden large generation) from normal operations
4. Use quantile regression to provide distribution forecasts for grid margin planning

## Summary
- Forecasted daily wind generation from Spain energy dataset
- High CV (0.5+) makes wind much harder than solar (CV ~0.2)
- Short-memory ML with recent lags performs competitively in the absence of NWP features
- Key lesson: statistical TS models have limited ceiling for wind; NWP integration is the path forward
*Notebook #20 of 50 — Time Series Forecasting Portfolio*